In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.9 Lagrange Points and the Restricted Three-Body Problem

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.9",
    title="Lagrange Points and the Restricted Three-Body Problem",
    blurb="Five points where a small body can ride along with two large ones: "
    "the effective potential of the rotating frame, the collinear and "
    "triangular equilibria, the mass-ratio threshold that makes L4 and "
    "L5 stable, and the tadpole orbits of the Trojan asteroids.",
    difficulty="advanced",
    estimate="90–120 min",
)

## Notebook overview

Three gravitating bodies have no general closed-form orbit: the three-body
problem is the classic edge where analytical mechanics runs out of formulae and
computation takes over. But one special case is both tractable and useful. Let
two massive bodies (the **primaries**, say the Earth and the Moon) circle their
common centre of mass, and watch a third body so light that it pulls on nobody.
Where can that third body sit so that, seen from the rotating frame of the
primaries, it does not move at all? There are exactly five such points, found by
Euler and Lagrange, and they are not a curiosity: the James Webb Space Telescope
parks at one of them, and thousands of Trojan asteroids librate around two more.

The trick that makes the five points visible is the one from [§2.4](central-force.ipynb), raised
to two dimensions and given a twist. In the co-rotating frame the motion is
governed by an **effective potential** Ω, the genuine gravity of the two
primaries plus a centrifugal term from the rotation. Its stationary points are
the equilibria, and we hunt them exactly as [§2.4](central-force.ipynb) hunted turning points: as roots,
found with the bracket-discovering solver now living in `ecp.mechanics`. Whether
each equilibrium is stable is a small-oscillation question ([§2.7](small-oscillations.ipynb)), with
one beautiful surprise: the triangular points L4 and L5 are *minima* of Ω yet
dynamically unstable (the rotating-frame force runs up the Ω-gradient), and the
Coriolis force can still trap a body there, provided one primary is heavy
enough relative to the other.

We will map Ω, locate all five points, settle their stability and the mass-ratio
threshold that governs it, and then watch the payoff: a test mass started just off
L4 traces a slow **tadpole** orbit instead of falling away, exactly as Jupiter's
Trojans do. That tadpole genuinely moves, so it earns an animation; the potential
map, the point map, and the stability curve are static, as such things should be.

> **How to read the checks.** Each exercise ends with a `validate` call against an
> independent fact: a finite-difference gradient, a vanishing force, an
> equilateral distance, a conserved Jacobi constant. A ✓ is strong evidence; a ✗
> is a prompt to *locate the discrepancy*, not a verdict.

> **Scope.** A working application, not a treatise on celestial mechanics. The
> standard references are Murray & Dermott, *Solar System Dynamics*
> {cite}`murraydermott`, and Szebehely's *Theory of Orbits* {cite}`szebehely`;
> the mechanics background is Nolting {cite}`nolting1`,{cite}`nolting2`.

## Theory in brief

### The restricted three-body problem

Two primaries of masses $m_1 \ge m_2$ orbit their barycentre on circles; a third
body of negligible mass moves in their combined gravity without affecting them.
We work in the **co-rotating frame** in which the primaries are fixed, and
nondimensionalise so that the total mass, the separation, and the angular velocity
are all $1$. With the mass ratio

$$
\mu = \frac{m_2}{m_1 + m_2}, \qquad 0 < \mu \le \tfrac12,
$$

the heavier primary (mass $1-\mu$) sits at $(-\mu, 0)$ and the lighter (mass
$\mu$) at $(1-\mu, 0)$. The Earth–Moon system, our running example, has
$\mu = 0.01215$.

### The effective potential

In the rotating frame the planar motion obeys (Murray & Dermott
{cite}`murraydermott`, ch. 3, derive the frame transformation in full)

```{math}
:label: eq-omega
\ddot x - 2\dot y = \frac{\partial\Omega}{\partial x}, \qquad
\ddot y + 2\dot x = \frac{\partial\Omega}{\partial y}, \qquad
\Omega(x,y) = \tfrac12\big(x^2+y^2\big) + \frac{1-\mu}{r_1} + \frac{\mu}{r_2},
```

where $r_1, r_2$ are the distances to the two primaries. The $\tfrac12(x^2+y^2)$
term is the centrifugal contribution of the rotating frame and the other two are
gravity; the $2\dot y$, $2\dot x$ terms are the Coriolis force, which depends on
the velocity and so does no work. Equilibria of the rotating-frame motion are the
stationary points $\nabla\Omega = 0$: this is the effective potential of [§2.4](central-force.ipynb),
now in two dimensions and carrying a centrifugal term.

### The five Lagrange points

Solving $\nabla\Omega = 0$ ({eq}`eq-omega`) gives five equilibria
({eq}`eq-lagrange`). Three are **collinear**, on the $y=0$ axis where
$\partial\Omega/\partial x = 0$: $L_1$ between the primaries, $L_2$ beyond the
lighter one, $L_3$ beyond the heavier one. Two are **triangular**, $L_4$ and $L_5$,
forming equilateral triangles with the primaries at

```{math}
:label: eq-lagrange
L_{4,5} = \left(\tfrac12 - \mu,\ \pm\tfrac{\sqrt3}{2}\right),
```

each a unit distance from *both* primaries.

### Stability and the Routh criterion

Linearising the rotating-frame equations about a Lagrange point gives a small-
oscillation problem ([§2.7](small-oscillations.ipynb)) with a twist: the Hessian of $\Omega$ supplies
the restoring terms and the Coriolis force couples the coordinates. The collinear
points are **saddles**, hence unstable. The triangular points are stable only when
the primaries are lopsided enough, the **Routh criterion** (Exercise 5 traces the
threshold numerically; Murray & Dermott {cite}`murraydermott` carry the
linearisation to the closed form)

```{math}
:label: eq-routh
\mu < \mu_{\rm crit} = \tfrac12\Big(1 - \sqrt{\tfrac{23}{27}}\Big) \approx 0.0385,
\qquad \text{equivalently}\quad \frac{m_1}{m_2} > 24.96 .
```

What makes this surprising is that $L_4$ and $L_5$ are *minima* of $\Omega$, yet
unstable: the rotating-frame acceleration runs *up* the gradient ($+\nabla\Omega$),
so with no rotation a body rolls off. The velocity-dependent Coriolis force curves
its path back, trapping it, provided the mass ratio clears the threshold.

### The Jacobi constant

The rotating frame does work, so energy is not conserved; but the combination

```{math}
:label: eq-jacobi
C = 2\Omega(x,y) - v^2, \qquad v^2 = \dot x^2 + \dot y^2,
```

is. This **Jacobi constant** is the one integral of motion of the problem, and the
curve $2\Omega = C$ (zero velocity) bounds the region the test body can reach.

### Beyond gravity

"Equilibrium in the effective potential of a rotating or interacting system" is a
general idea, not only a gravitational one. The same stationary-point-plus-Hessian
stability analysis locates equilibrium configurations and transition states on
molecular potential-energy surfaces, and governs many-body equilibria more broadly
(a forward link to the later volumes). Here it is gravity, with two famous hooks:
JWST parks at the Sun–Earth $L_2$, and Jupiter's Trojan asteroids occupy its
$L_4$ and $L_5$.

## Setup

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.integrate import solve_ivp

from ecp import validate
from ecp.animate import show
from ecp.mechanics import bracketed_roots

MU = 0.01215  # Earth–Moon-like mass ratio, the running example

# Symbolic effective potential Ω(x, y; μ) and its first/second derivatives, then
# lambdified for fast numerics. Using SymPy here (the §2.1 habit) keeps the gradient
# and Hessian exact and free of hand-differentiation slips.
_x, _y, _mu = sp.symbols("x y mu", real=True)
_r1 = sp.sqrt((_x + _mu) ** 2 + _y**2)  # distance to the heavier primary at (−μ, 0)
_r2 = sp.sqrt(
    (_x - 1 + _mu) ** 2 + _y**2
)  # distance to the lighter primary at (1−μ, 0)
_Omega = sp.Rational(1, 2) * (_x**2 + _y**2) + (1 - _mu) / _r1 + _mu / _r2
_Ox, _Oy = sp.diff(_Omega, _x), sp.diff(_Omega, _y)
_Oxx, _Oxy, _Oyy = sp.diff(_Ox, _x), sp.diff(_Ox, _y), sp.diff(_Oy, _y)

Omega = sp.lambdify((_x, _y, _mu), _Omega, "numpy")
grad_Omega = sp.lambdify((_x, _y, _mu), [_Ox, _Oy], "numpy")
hessian_Omega = sp.lambdify((_x, _y, _mu), [[_Oxx, _Oxy], [_Oxy, _Oyy]], "numpy")


def primaries(mu=MU):
    """Positions of the two primary masses in the rotating frame.

    The heavier at (−μ, 0) and lighter at (1−μ, 0), the fixed sources of the
    effective potential.

    Parameters
    ----------
    mu : float, optional
        Mass ratio (default the module value).

    Returns
    -------
    tuple of numpy.ndarray
        The two primary positions.
    """
    return np.array([-mu, 0.0]), np.array([1.0 - mu, 0.0])


def rotating_rhs(_t, s, mu=MU):
    """Equations of motion in the co-rotating frame (eq-omega).

    Includes the centrifugal and Coriolis terms; its equilibria are the
    Lagrange points.

    Parameters
    ----------
    _t : float
        Time (unused).
    s : array_like
        State ``[x, y, vx, vy]``.
    mu : float, optional
        Mass ratio.

    Returns
    -------
    list
        The state derivative.
    """
    x, y, vx, vy = s
    Ox, Oy = grad_Omega(x, y, mu)
    return [vx, vy, 2.0 * vy + Ox, -2.0 * vx + Oy]


def jacobi_constant(x, y, vx, vy, mu=MU):
    """The Jacobi constant C = 2Ω − v^2 (eq-jacobi).

    The lone conserved quantity of the restricted three-body problem; it
    bounds the accessible region.

    Parameters
    ----------
    x, y, vx, vy : float or numpy.ndarray
        Position and velocity in the rotating frame.
    mu : float, optional
        Mass ratio.

    Returns
    -------
    float or numpy.ndarray
        The Jacobi constant.
    """
    return 2.0 * Omega(x, y, mu) - (vx**2 + vy**2)


def stability_eigenvalues(x0, y0, mu=MU):
    """Eigenvalues of the motion linearised about a point.

    Their real parts decide stability: a positive real part means an unstable (saddle) point.

    Parameters
    ----------
    x0, y0 : float
        Point about which to linearise.
    mu : float, optional
        Mass ratio.

    Returns
    -------
    numpy.ndarray
        The four eigenvalues.
    """
    (Oxx, Oxy), (_, Oyy) = hessian_Omega(x0, y0, mu)
    A = np.array(
        [
            [0.0, 0.0, 1.0, 0.0],
            [0.0, 0.0, 0.0, 1.0],
            [Oxx, Oxy, 0.0, 2.0],
            [Oxy, Oyy, -2.0, 0.0],
        ]
    )
    return np.linalg.eigvals(A)


L4_POINT = np.array([0.5 - MU, np.sqrt(3) / 2.0])  # triangular point (this μ)
L5_POINT = np.array([0.5 - MU, -np.sqrt(3) / 2.0])

## Exercise 1 — The effective potential of the rotating frame

Before locating anything we need the landscape itself. The effective potential
{eq}`eq-omega` is the landscape whose gradient drives the rotating-frame motion:
two towering gravitational spikes at the primaries (where $\Omega\to+\infty$), and
a centrifugal rise that grows like $\tfrac12 r^2$ far out. A rotating-frame "potential" makes sense precisely because
the Coriolis force does no work, so only Ω sets where the body can rest.

**This exercise.** With $\mu = 0.01215$, state $\Omega$ explicitly ({eq}`eq-omega`)
and map it ({numref}`fig-l3b-omega`). As a correctness check on the analytic
gradient we will rely on throughout, compare it at a generic point against a
finite-difference gradient.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    g_analytic,
    g_fd,
    "the analytic gradient of Ω matches finite differences",
    rtol=1e-6,
)

## Exercise 2 — The collinear points L1, L2, L3 (worked)

On the axis $y = 0$ the problem collapses to one dimension: a Lagrange point is a
root of $\partial\Omega/\partial x = 0$ ({eq}`eq-lagrange`). The function has
poles at the two primaries, so we search three clean intervals (between the
primaries, beyond the lighter, beyond the heavier) and let
`ecp.mechanics.bracketed_roots` (which scans each interval on a fine sub-grid for
sign changes, then refines each bracket) do the rest, a direct callback to the
turning-point search of [§2.4](central-force.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [dOmega_dx_axis(L1), dOmega_dx_axis(L2), dOmega_dx_axis(L3)],
    [0.0, 0.0, 0.0],
    "L1, L2, L3 are stationary points of Ω",
    atol=1e-9,
)

## Exercise 3 — The triangular points L4, L5 are equilateral

The remaining two equilibria sit off the axis. The claim is that
$L_{4,5} = (\tfrac12-\mu,\ \pm\tfrac{\sqrt3}{2})$ ({eq}`eq-lagrange`) are stationary
and that each is a *unit* distance from both primaries, so the body and the two
primaries form an equilateral triangle. This is exact for any $\mu$.

**This exercise.** Confirm numerically that $\nabla\Omega$ vanishes at $L_4$ and
that $r_1 = r_2 = 1$ there, then mark all five points on the potential map
({numref}`fig-l3b-points`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    [r1_L4, r2_L4],
    [1.0, 1.0],
    "L4 forms an equilateral triangle with the primaries (unit distances)",
    rtol=1e-9,
)
validate.close(
    grad_L4, [0.0, 0.0], "∇Ω vanishes at L4 (it is an equilibrium)", atol=1e-12
)

## Exercise 4 — Stability I: the collinear saddles

Knowing where the points are says nothing about whether a body *stays* there. We
settle that the way [§2.7](small-oscillations.ipynb) settled small oscillations: linearise the motion
about the point and read off the eigenvalues. The rotating-frame linearisation
carries the Hessian of $\Omega$ plus the Coriolis coupling, giving a $4\times4$
system. A purely imaginary spectrum means oscillation (stable); any eigenvalue
with a positive real part means runaway growth (unstable).

**This exercise.** Linearise about $L_1$ ({eq}`eq-routh`), assemble the $4\times4$
first-order system, and take its eigenvalues with `numpy.linalg.eigvals`; show one
has a positive real part: the collinear points are saddles.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    max_real_L1 > 0.0,
    "the collinear point L1 is linearly unstable (a saddle)",
    f"max Re(λ) = {max_real_L1:.4f}",
)

## Exercise 5 — Stability II: the triangular points and the Routh criterion

This is the centrepiece, and the surprise. $L_4$ and $L_5$ are *minima* of
$\Omega$ — but the rotating-frame force runs up the $\Omega$-gradient, so drop a
body there with the rotation switched off and it rolls away. Yet
with rotation the velocity-dependent Coriolis force bends its path into a closed
loop, *provided the primaries are lopsided enough*. The threshold is the Routh
criterion {eq}`eq-routh`, $\mu_{\rm crit} = \tfrac12(1-\sqrt{23/27}) \approx
0.0385$, equivalently a mass ratio $m_1/m_2 > 24.96$.

**This exercise.**

1. Compute $\mu_{\rm crit}$.
2. Show $L_4$ is stable for the Earth–Moon $\mu = 0.01215$ and unstable for a
   fictitious $\mu = 0.06$ above the threshold.
3. Trace the stability boundary by plotting, against $\mu$, the largest real
   part of the four `numpy.linalg.eigvals` eigenvalues of the linearised
   system at $L_4$ ({numref}`fig-l3b-stability`).

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    mu_crit,
    0.5 * (1 - np.sqrt(23 / 27)),
    "the Routh critical mass ratio is ½(1−√(23/27)) ≈ 0.0385",
    rtol=1e-9,
)
validate.check(
    stable_em < 1e-6 and unstable_hi > 1e-3,
    "L4 is stable below μ_crit (Earth–Moon) and unstable above it (μ=0.06)",
    f"Re(λ): μ=0.01215 → {stable_em:.2e},  μ=0.06 → {unstable_hi:.3f}",
)

## Exercise 6 — Tadpole orbits near L4 (worked animation)

Now the payoff, and it genuinely moves. A test mass started just off $L_4$, at
rest in the rotating frame, does not fall into a primary and does not fly off: it
*librates* around $L_4$ on a long, curved **tadpole** orbit, the path swinging out
toward $L_3$ and back. This is exactly what Jupiter's Trojan asteroids do at its
$L_4$ and $L_5$. A still frame cannot convey the slow circulation, so we animate it.

**This exercise.** Start a test mass at $L_4 + (0.001, 0.001)$, at rest in the
rotating frame, with $\mu = 0.01215$. Using `scipy.integrate.solve_ivp` (its
default `RK45`) with tolerances `rtol=1e-10`, `atol=1e-12` and `dense_output=True`
(sampled at 600 points), integrate the rotating-frame equations of motion
{eq}`eq-omega` from $t = 0$ to $t = 100$; then animate the path, with the
primaries and $L_4$ marked. The validation confirms the mass stays bounded near
$L_4$ (a stable tadpole), the physics the animation shows.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    max_dev_L4 < 0.1,
    "the test mass stays bounded near L4: a stable tadpole orbit",
    f"max deviation = {max_dev_L4:.3f} over t∈[0,100]",
)

## Exercise 7 — The Jacobi constant and the unstable contrast (student exercise)

The rotating frame does work, so the test mass's energy is not conserved. One
combination is: the Jacobi constant {eq}`eq-jacobi`, $C = 2\Omega - v^2$. Checking
that it stays flat along the tadpole orbit is an independent gate on the whole
integration, and the contrast with an unstable point makes the stability concrete.

1. For the $L_4$ orbit of Exercise 6, compute $C(t)$ along the trajectory
   (the `jacobi_constant` helper) and confirm it is conserved even though the
   energy is not.
2. With `scipy.integrate.solve_ivp` (same `rtol=1e-10`, `atol=1e-12`),
   integrate a test mass started at $L_1 + (0.001, 0.001)$ (at rest, same
   $\mu$) to $t = 20$ and show it diverges quickly, in contrast to the
   bounded $L_4$ orbit.
3. Note what the conservation of $C$ buys: the zero-velocity curve
   $2\Omega = C$ bounds where the body can go.

Because the checks test the *trajectory data* (a conserved $C$, a growing
deviation), a ✗ points at the integration or the initial condition, not at any
drawing.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.conserved(
    C_series,
    "the Jacobi constant is conserved along the L4 trajectory",
    rel_drift=1e-6,
)
validate.check(
    dev_L1.max() > 0.1 and max_dev_L4 < 0.1,
    "L1 is unstable (the test mass leaves its neighbourhood) while L4 stays bounded",
    f"L1 deviation {dev_L1.max():.2f} (escapes) vs L4 deviation {max_dev_L4:.3f} (bounded)",
)

## Exercise 8 — Where the Lagrange points live in the real solar system (synthesis)

The five points are not a textbook abstraction. The collinear $L_2$ of the
Sun–Earth system, $1.5$ million km beyond the Earth, is where the James Webb Space
Telescope parks: a single vantage with the Sun, Earth, and Moon all behind it.
Because $L_2$ is a saddle (Exercise 4), JWST does not sit *at* it but flies a halo
orbit around it and burns a little fuel every few weeks to stay (station-keeping).
The triangular points are different: stable, and so they *collect* material. At
Jupiter's $L_4$ and $L_5$ ride the **Trojan asteroids**, thousands of them, held
for the age of the solar system because the Sun–Jupiter mass ratio sits far above
the Routh threshold of 24.96. The same "equilibrium in an effective potential"
idea reaches well beyond gravity, to the stationary points and saddles of
molecular potential-energy surfaces and to many-body equilibria generally.

**This exercise.** Confirm the Sun–Jupiter mass ratio clears the Routh threshold,
so its Trojan points are stable.

In [ ]:
# (solution hidden on the public site)


### Validation 8

In [ ]:
validate.check(
    ratio_sun_jupiter > 24.96,
    "the Sun–Jupiter mass ratio exceeds 24.96, so the Trojan points are stable",
    f"ratio = {ratio_sun_jupiter:.1f}",
)

## Notebook summary

- The effective potential of the co-rotating frame, the collinear points $L_1,L_2,L_3$
  (roots of $\partial\Omega/\partial x$ on the axis, by bracketed sign-change scan) and
  the triangular $L_4,L_5$ shown to be exactly equilateral.
- Their stability: the collinear points are saddles; the triangular points are stable when the
  mass ratio clears the **Routh** threshold ($\approx 0.0385$). Tadpole orbits near $L_4$, the
  Jacobi constant bounding the motion, and where the Lagrange points sit in the real solar system.

## Outlook

- **Halo and Lissajous orbits** about the collinear points, and the station-keeping
  that JWST needs because $L_2$ is unstable.
- **Zero-velocity (Hill) curves** from the Jacobi constant: the forbidden regions
  that open and close as $C$ changes.
- **Weak stability boundaries and low-energy transfers**, the "interplanetary
  superhighway" that threads the collinear points' invariant manifolds.
- **Forward link.** The effective-potential-equilibrium picture, stationary point
  plus Hessian stability, reappears for saddle points on molecular potential-energy
  surfaces, the same analysis in a different physical setting.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()